# EX-0625 — Démonstration G15
Cliphton Marseille et Igor Gaitan · Cerise sur le ghetto

1. Lancer l’installation (▶️).
2. Lancer la configuration (▶️) et saisir la clé API du cours si demandée. Ne pas la mettre dans le code.
3. Lancer le test (▶️). Par défaut, Lucas n’a pas d’email.
4. Pour le deuxième essai, changer uniquement `CHOIX_TEST = 2` en `CHOIX_TEST = 3`, puis relancer la dernière cellule.

La configuration utilise OpenRouter et le modèle qwen/qwen3.8-flash par défaut. Si votre clé dépend d’un autre service, adaptez la configuration avant de lancer. Aucun résultat IA n’est prérempli. Les réponses sont à vérifier après exécution.

Assistance ChatGPT/Codex déclarée. Source : notebook du cours `03_Structured_Outputs.ipynb`, EX-0625.


In [ ]:
%pip install -q "pydantic>=2,<3" openai python-dotenv


## Configuration de l’accès à l’IA
À exécuter une fois par session. La saisie de la clé est masquée.


In [ ]:
import os, json
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError, conint
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
cle = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")
if not cle:
    from getpass import getpass
    cle = getpass("Clé API du cours (saisie masquée) : ")
    if not cle.strip():
        raise RuntimeError("La clé API du cours est nécessaire pour appeler le modèle.")
client = OpenAI(api_key=cle, base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"), max_retries=0, timeout=60)
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "qwen/qwen3.8-flash")
def add_additional_properties_false(schema: dict) -> dict:
    """
    Ajoute récursivement additionalProperties: False à tous les objets du schema
    ET s'assure que tous les champs sont dans required (requis par OpenAI).
    
    OpenAI Structured Outputs exige:
    1. additionalProperties: False sur tous les objets
    2. Tous les champs dans 'required' (même ceux avec defaults)
    """
    if not isinstance(schema, dict):
        return schema
    
    # Traiter les définitions $defs d'abord (Pydantic 2.x les génère)
    if "$defs" in schema:
        for def_name, def_schema in schema["$defs"].items():
            add_additional_properties_false(def_schema)
    
    # Si c'est un objet, ajouter additionalProperties: False
    # ET s'assurer que tous les champs sont dans required
    if schema.get("type") == "object":
        schema["additionalProperties"] = False
        # Ajouter tous les champs de properties dans required
        if "properties" in schema:
            all_props = list(schema["properties"].keys())
            schema["required"] = all_props
            # Traiter récursivement les propriétés
            for prop_name, prop_schema in schema["properties"].items():
                add_additional_properties_false(prop_schema)
    
    # Traiter les items d'arrays
    if schema.get("type") == "array" and "items" in schema:
        add_additional_properties_false(schema["items"])
    
    # Traiter anyOf, allOf, oneOf
    for key in ["anyOf", "allOf", "oneOf"]:
        if key in schema:
            for item in schema[key]:
                add_additional_properties_false(item)
    
    return schema

def extract_structured(prompt: str, model_class: type[BaseModel], model: str = None):
    """
    Fonction utilitaire pour extraction structurée avec gestion d'erreurs.

    Args:
        prompt: Prompt utilisateur
        model_class: Classe Pydantic pour validation
        model: Modèle OpenAI à utiliser (défaut: DEFAULT_MODEL)

    Returns:
        Instance validée de model_class ou None en cas d'erreur
    """
    if model is None:
        model = DEFAULT_MODEL
    
    try:
        # Générer le schema et ajouter additionalProperties: False (requis par OpenAI)
        schema = model_class.model_json_schema()
        schema = add_additional_properties_false(schema)

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": model_class.__name__.lower(),
                    "strict": True,
                    "schema": schema
                }
            }
        )
        return model_class.model_validate_json(response.choices[0].message.content)
    except ValidationError as e:
        print(f"Erreur de validation Pydantic: {e}")
        return None
    except Exception as e:
        print(f"Erreur API: {e}")
        return None



## Test à présenter
Le programme vérifie les cinq coordonnées par rapport à une référence annoncée avant l’appel. Le compteur des champs absents dépend du test. Une erreur d’extraction est distincte d’une information absente.

Cas 0 : Ines, complet. Cas 1 : Ines, trois informations absentes. Cas 2 : Lucas, email absent. Cas 3 : Lucas, email ajouté.


In [ ]:
from typing import Optional, Literal, Union
from collections import Counter
from pydantic import BaseModel, Field
import json
import unicodedata

class ContactExtrait(BaseModel):
    nom: str
    telephone: str
    email: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")
    entreprise: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")
    ville: Optional[str] = Field(default=None, description="null si l'information n'apparait pas dans le texte")

texte_complet = ("Bonjour, je m'appelle Ines Benali, joignable au 06 98 76 54 32. "
                 "Email : ines.benali@exemple.fr, societe Atelier Numerique, Lyon. Je souhaite un devis.")
texte_lacunaire = "Bonjour, je m'appelle Ines Benali, joignable au 06 98 76 54 32. Je souhaite un devis."
texte_lucas = "Bonjour, je suis Lucas Martin de l'entreprise Nova, à Bordeaux. Vous pouvez me rappeler au 06 12 34 56 78. Je souhaite un devis."
consigne = "Extrais uniquement les informations explicites du texte, sans inference. Utilise null pour les informations absentes. Texte : "

# La référence est définie AVANT l'appel à l'IA, à partir de chaque texte.
# Pour ajouter un autre message, indiquer aussi ses valeurs attendues.
cas_tests = [
    ("Ines : message complet", texte_complet,
     dict(nom="Ines Benali", telephone="06 98 76 54 32", email="ines.benali@exemple.fr", entreprise="Atelier Numerique", ville="Lyon")),
    ("Ines : trois informations absentes", texte_lacunaire,
     dict(nom="Ines Benali", telephone="06 98 76 54 32", email=None, entreprise=None, ville=None)),
    ("Lucas : email absent", texte_lucas,
     dict(nom="Lucas Martin", telephone="06 12 34 56 78", email=None, entreprise="Nova", ville="Bordeaux")),
    ("Lucas : email ajouté", texte_lucas + " Mon email est lucas.martin@exemple.fr.",
     dict(nom="Lucas Martin", telephone="06 12 34 56 78", email="lucas.martin@exemple.fr", entreprise="Nova", ville="Bordeaux")),
]

def normaliser(valeur):
    if valeur is None:
        return None
    texte = unicodedata.normalize("NFKD", valeur.casefold())
    return "".join(c for c in texte if c.isalnum())

def verifier_contact(resultat, reference):
    if resultat is None:
        return {"extraction_reussie": False}
    valeurs = resultat.model_dump()
    absents_attendus = [champ for champ, valeur in reference.items() if valeur is None]
    return {
        "extraction_reussie": True,
        "valeurs": valeurs,
        "champs_renseignes": sum(v is not None for v in valeurs.values()),
        "absents_attendus": absents_attendus,
        "absents_corrects": sum(valeurs[c] is None for c in absents_attendus),
        "champs_conformes": {c: normaliser(valeurs[c]) == normaliser(v) for c, v in reference.items()},
    }

# Un seul cas par lancement pour la démonstration, sans relancer tout le cours.
# Choisir 0 ou 1 pour l'exercice initial, puis 2 et 3 pour Lucas.
CHOIX_TEST = 2
etiquette, texte, reference = cas_tests[CHOIX_TEST]
print("TEST :", etiquette)
print("MESSAGE :", texte)
resultat = extract_structured(consigne + texte, ContactExtrait)
bilan = verifier_contact(resultat, reference)
if not bilan["extraction_reussie"]:
    print("ECHEC D'EXTRACTION : vérifier la configuration ou l'erreur affichée plus haut.")
else:
    print("REPONSE DE L'IA :")
    print(json.dumps(bilan["valeurs"], ensure_ascii=False, indent=2))
    print("Champs renseignés :", bilan["champs_renseignes"])
    total_absents = len(bilan["absents_attendus"])
    if total_absents:
        print("Champs attendus absents :", ", ".join(bilan["absents_attendus"]))
        print(f"Absences respectées : {bilan['absents_corrects']}/{total_absents}")
    else:
        print("Aucune information attendue absente pour ce test.")
    print("VERIFICATION DES CINQ CHAMPS :")
    for champ, conforme in bilan["champs_conformes"].items():
        print(champ, ":", "OK" if conforme else "A VERIFIER", "/ attendu :", repr(reference[champ]))
    print("Tous les champs conformes :", all(bilan["champs_conformes"].values()))
# Comparaison à une référence explicite, avec normalisation simple de l'écriture.
# Elle ne remplace pas une vérification sémantique générale.
